# 2 · Claude Code I: AI as a Coding Copilot

**Outcome of this session:** your first complete finance Python project, a Comparable Company Analysis tool, published on your own GitHub.

**In this notebook you will:**

- Clean a real financial dataset with every correction visible
- Build a Comparable Company Analysis tool, guided by Claude Code
- Compute and chart the standard valuation multiples
- Publish your first finance project to GitHub


## The working rhythm
With Claude Code you are the analyst in charge; the model is a fast assistant:

1. **Ask small.** One function, one fix, one chart at a time.
2. **Read before you run.** Can't explain a line? Ask Claude to explain it.
3. **Verify one number by hand.** For every table you produce, pick one cell and confirm it on a calculator.
4. **Commit at every green moment.** Small commits make each step reversible.

**How to use Claude Code in this notebook:** select an exercise's docstring, press `Option+K` (Mac) / `Alt+K` (Windows): the file and lines land in the ✱ panel: then ask *"implement this"*. Read the diff it proposes before accepting. That loop, hundreds of times, is the career skill.

**Pandas in one paragraph:** a `DataFrame` is the analyst's table. `pd.read_csv` loads it; columns are vectors, so `df["a"] / df["b"]` computes a whole ratio column at once; you don't memorize pandas: you *specify* what you want and *verify* what you get.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: a financial data pipeline on messy data
Real data arrives broken. This file holds **real SEC-filed fundamentals** for 10 big-tech companies with 8 realistic defects injected (documented in `session-02-coding-copilot/data/README.md`). Rule one: **inspect before you fix.**

In [ ]:
import pandas as pd
pd.options.display.float_format = "{:,.1f}".format

raw = pd.read_csv(ROOT / "session-02-coding-copilot" / "data" / "tech_financials_messy.csv")
raw.head(13)

The problems to spot:

- headers with spaces and symbols
- revenue stored as text, with thousands separators (`"416,161.0"`)
- a `Unit` column where one company reports in **billions**
- a duplicated row
- `orcl ` in lowercase, with a trailing space
- missing values
- a junk `TOTAL` row
- a stray `Notes` column

Now we fix **each one explicitly**: visible, reviewable, no black box:

In [ ]:
df = raw.copy()

# 1. headers -> snake_case
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^\w]+", "_", regex=True).str.strip("_"))
df = df.rename(columns={"revenue_fy_m": "revenue_m", "revenue_fy_1_m": "revenue_prior_m",
                        "revenue_fy_2_m": "revenue_prior2_m", "op_income_m": "operating_income_m",
                        "d_a_m": "d_and_a_m", "price": "price_usd"})

# 2. junk out: TOTAL row, notes column
df = df[df["ticker"].str.strip().str.upper() != "TOTAL"].drop(columns=["notes"])

# 3. tickers normalized, duplicates dropped
df["ticker"] = df["ticker"].str.strip().str.upper()
df = df.drop_duplicates(subset="ticker", keep="first")

# 4. text numbers -> real numbers
num_cols = ["revenue_m", "revenue_prior_m", "revenue_prior2_m", "operating_income_m",
            "net_income_m", "d_and_a_m", "cash_m", "total_debt_m", "shares_m", "price_usd"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ""), errors="coerce")

# 5. the silent 1000x error: one row is in $B - rescale to $M
in_b = df["unit"] == "USD_B"
df.loc[in_b, [c for c in num_cols if c != "price_usd"]] *= 1000
df = df.drop(columns=["unit"]).reset_index(drop=True)

print(f"clean: {len(df)} companies (rescaled {int(in_b.sum())} from $B)")
df[["ticker", "revenue_m", "operating_income_m", "net_income_m"]]

In [ ]:
# quick KPIs + a look at the shape of the industry
df["revenue_growth"] = df["revenue_m"] / df["revenue_prior_m"] - 1
df["op_margin"] = df["operating_income_m"] / df["revenue_m"]

ax = df.sort_values("op_margin", ascending=False).plot.bar(
    x="ticker", y="op_margin", legend=False, figsize=(9, 3.5),
    color=["#c0392b" if v < 0 else "#2a9d5c" for v in df.sort_values("op_margin", ascending=False)["op_margin"]])
ax.set_title("Operating margin - latest fiscal year (real SEC data)")
ax.axhline(0, color="black", lw=0.8);

One detail of the output deserves attention: **Intel's operating margin is negative**, and the pipeline handled it without any special treatment. The figure is genuine, not planted: Intel reported an operating loss of $2.2 billion for fiscal 2025 in its own 10-K, so the dataset simply reflects the filing. No cleaning step filtered the company out, and the arithmetic simply produced a negative percentage.

This matters because real datasets always include a company in a bad year. Code written on the assumption that every value is positive either breaks when the assumption fails or, worse, silently drops the company. The multiples you build next meet the same issue in a sharper form: a ratio over negative earnings is meaningless, and the professional convention is to report it as **n.m.** rather than publish a negative multiple.

This concludes the demonstration. The lab is next, and you build the analysis itself.

---

## Part B: LAB: the Comparable Company Analysis tool

Definitions you need (approximations documented in the data README):

- **EBITDA ≈ operating income + depreciation & amortization.** A proxy for the cash the operations generate, before financing costs, taxes and the accounting choices behind depreciation schedules. It makes companies with different capital structures comparable.
- **Market capitalization = shares outstanding × share price.** The market value of the equity alone.
- **Enterprise value (EV) = market cap + total debt − cash.** The value of the whole business: what acquiring the equity and assuming the debt, net of the cash acquired, would cost.
- **EV/EBITDA = enterprise value ÷ EBITDA.** The standard comparison multiple: the price of the whole business per unit of operating earnings. It is capital-structure-neutral, which is why it, and not P/E, anchors most comparable analyses.
- **EV/Sales = enterprise value ÷ revenue.** The fallback multiple when EBITDA is negative or unrepresentative; revenue is rarely negative, so it is almost always computable.
- **P/E = market capitalization ÷ net income.** The price of the equity per unit of net profit. Sensitive to leverage and one-off items, which is why it complements rather than replaces EV/EBITDA.
- **A multiple over a negative denominator is meaningless.** Return `NaN` and report it as **n.m.**; never publish a negative multiple.

We work on the clean dataset:

In [ ]:
import numpy as np
comps = pd.read_csv(ROOT / "session-02-coding-copilot" / "data" / "tech_financials.csv")
comps.head(3)

### Exercise 1: growth and margins

Add five columns: `revenue_growth_1y` (vs prior), `revenue_cagr_2y` (two-year compound growth: `(rev/rev_prior2)**0.5 - 1`), `ebitda_m`, `op_margin`, `ebitda_margin`.

In [ ]:
def add_growth_and_margins(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
### START CODE HERE ###
    df["revenue_growth_1y"] = df[None] / df[None] - 1
    df["revenue_cagr_2y"] = (df[None] / df[None]) ** None - 1    # exponent for a TWO-year CAGR?
    df["ebitda_m"] = df[None] + df[None]                         # EBITDA = which two columns?
    df["op_margin"] = df[None] / df["revenue_m"]
    df["ebitda_margin"] = df[None] / df["revenue_m"]
### END CODE HERE ###
    return df

comps = add_growth_and_margins(comps)
comps[["ticker", "revenue_growth_1y", "ebitda_margin"]].round(3)

In [ ]:
# ✅ self-check: run me
r = comps.set_index("ticker")
assert "revenue_growth_1y" in comps and "ebitda_margin" in comps, "missing columns"
assert abs(r.loc["AAPL", "revenue_growth_1y"] - (r.loc["AAPL", "revenue_m"] / r.loc["AAPL", "revenue_prior_m"] - 1)) < 1e-9
assert r.loc["INTC", "op_margin"] < 0, "Intel's operating margin should be negative - don't 'fix' real data"
assert (comps["revenue_cagr_2y"].abs() < 1.5).all(), "CAGR out of range - check the exponent (2-year: **0.5)"
print("All checks passed ✅")

### Exercise 2: valuation multiples

Add `mcap_m`, `ev_m`, `ev_ebitda`, `ev_sales`, `pe`. Negative EBITDA or net income ⇒ `np.nan` (hint: `np.where(cond, value, np.nan)`).

In [ ]:
def add_multiples(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
### START CODE HERE ###
    df["mcap_m"] = df[None] * df[None]                            # market cap = shares x ?
    df["ev_m"] = df["mcap_m"] + df[None] - df[None]               # EV = mcap + debt - cash
    df["ev_ebitda"] = np.where(df["ebitda_m"] > 0, df[None] / df[None], np.nan)
    df["ev_sales"] = df["ev_m"] / df[None]
    df["pe"] = np.where(df[None] > 0, df["mcap_m"] / df[None], np.nan)
### END CODE HERE ###
    return df

comps = add_multiples(comps)
comps[["ticker", "ev_ebitda", "ev_sales", "pe"]].round(1)

In [ ]:
# ✅ self-check: run me
r = comps.set_index("ticker")
assert np.isnan(r.loc["INTC", "pe"]), "Intel lost money: P/E must be NaN (n.m.), never negative"
assert 5 < r.loc["AAPL", "ev_ebitda"] < 60, "AAPL EV/EBITDA looks wrong - check EV = mcap + debt - cash"
assert (comps["ev_ebitda"].dropna() > 0).all(), "no negative multiples allowed"
print("All checks passed ✅")
print("\nNow verify Apple's EV/EBITDA on a calculator, from its CSV row: one cell of every table, confirmed by hand.")

### Exercise 3: the summary an analyst could actually use

Return `ticker, revenue_m, revenue_growth_1y, ebitda_margin, ev_ebitda, ev_sales, pe`, sorted by `ev_ebitda` ascending (NaN last), plus a final `MEDIAN` row with column medians.

In [ ]:
def build_summary(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["ticker", "revenue_m", "revenue_growth_1y", "ebitda_margin", "ev_ebitda", "ev_sales", "pe"]
### START CODE HERE ###
    summary = df[cols].sort_values(None, na_position="last")      # sort by which multiple?
    median = summary.drop(columns="ticker").median(numeric_only=True)
    summary = pd.concat([summary, pd.DataFrame([{"ticker": None, **median.to_dict()}])],
                        ignore_index=True)                        # label for the final row?
### END CODE HERE ###
    return summary

summary = build_summary(comps)
summary.round(2)

In [ ]:
# ✅ self-check: run me
assert summary.iloc[-1]["ticker"] == "MEDIAN", "last row must be the MEDIAN"
assert len(summary) == len(comps) + 1
v = summary["ev_ebitda"].dropna().iloc[:-1]
assert (v.values == sorted(v.values)).all(), "sort by ev_ebitda ascending"
print("All checks passed ✅")

OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
summary.to_csv(OUTD / "comps_summary.csv", index=False)
print("Saved outputs/comps_summary.csv - this file goes in your portfolio.")

### The chart: multiples at a glance

A table answers precise questions; a chart shows who stands out. This one is **interactive**: hover a bar for the company's growth and margin, drag to zoom, double-click to reset. (Plotly draws interactive charts; seaborn and matplotlib, which the course also installs, draw static ones.)

In [ ]:
try:
    import plotly.express as px
except ModuleNotFoundError:          # first run on an env set up before plotly joined the course
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
    import plotly.express as px

plot_df = comps.dropna(subset=["ev_ebitda"]).sort_values("ev_ebitda")
fig = px.bar(
    plot_df, x="ticker", y="ev_ebitda",
    hover_data={"revenue_growth_1y": ":.1%", "ebitda_margin": ":.1%", "ev_ebitda": ":.1f"},
    labels={"ev_ebitda": "EV/EBITDA (x)", "ticker": ""},
    title="EV/EBITDA by company (negative-EBITDA companies excluded, hence no Intel)",
)
fig.add_hline(y=plot_df["ev_ebitda"].median(), line_dash="dash", annotation_text="median")
fig.show()

## Part C: publish to GitHub
Your comps tool is a project. In the **terminal** (not this notebook):

```bash
git init && git add . && git commit -m "Comps tool: first working version, AAPL multiple hand-verified"
gh repo create my-finance-toolkit --private --source . --push
```

(No `gh`? github.com → New repo → follow "push an existing repository". Cheatsheet: `cheatsheets/git-github-for-finance.md`.)

## Deliverable checklist

- [ ] All three ✅ checks green, with at least one exercise implemented via Claude Code (`Option/Alt+K` on the docstring)
- [ ] Apple's EV/EBITDA hand-verified on a calculator
- [ ] `outputs/comps_summary.csv` exists; Intel shows NaN P/E
- [ ] Repo pushed to GitHub with ≥2 commits
- [ ] Stretch: scatter `revenue_growth_1y` vs `ev_ebitda`: is growth priced in?

**Next:** `03-debugging-earnings.ipynb`: a valuation model that is wrong on purpose.